In [ ]:
# ── Colab setup: clone repo and install dependencies ──
import os, sys

if not os.path.exists('/content/repo'):
    !git clone https://github.com/joshgreenwa/Graph-Transformer-Attention-Interpretability.git /content/repo

if '/content/repo' not in sys.path:
    sys.path.insert(0, '/content/repo')

os.chdir('/content/repo')

# Match dependency setup used in generate_figures.ipynb
%pip install -r /content/repo/requirements.lock.txt
%pip install -i https://pypi.org/simple rdkit
!pip install numpy==1.26.4
!pip install ogb
!pip install scipy scikit-learn

print('Setup complete')


# Temporary Per-Query Score Experiments

This notebook is **intentionally separate** from the main figure-generation pipeline.

Goals:
1. Compute and visualise **per-query** key-permutation scores on one graph.
2. Compute and visualise mean per-query scores for **oxygen query nodes** across graphs.
3. Compare per-query key-permutation scores against **normalised KL divergence** scores.


In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from rdkit import Chem
from scipy.stats import pearsonr, spearmanr

from graph_interp.config import set_seed, apply_plot_defaults
from graph_interp.data import load_dataset, load_smiles_pool, build_batch_from_smiles
from graph_interp.extraction import load_model, extract_attention, node_only_conditional
from graph_interp.metrics.scores import sample_transpositions, cossim_lastdim, _swap_perm, aggregate_scores
from graph_interp.metrics.kl import _normalized_kl, aggregate_kl_scores
from graph_interp.metrics.cls import score_cls_struct_sym
from graph_interp.plots.section_i import (
    fig39_structural_dom_vs_kl_diff,
    fig40_kl_disagreement,
    fig41_signed_bias,
)

apply_plot_defaults()
set_seed(42)

model, device = load_model()
ds = load_dataset()
smiles_pool = load_smiles_pool(ds)
print(f"Loaded model on {device}. SMILES pool: {len(smiles_pool)}")


In [ ]:
@torch.no_grad()
def score_layer_struct_sym_per_query(
    dot_all: torch.Tensor,
    bias_all: torch.Tensor,
    A_all: torch.Tensor,
    num_perms: int = 64,
    tau: float = 0.02,
    seed: int = 0,
    query_indices: list[int] | None = None,
):
    """Per-query structural/symbolic scores for one layer.

    Returns:
      s_struct: [H, Q]
      s_sym:    [H, Q]
      q_idx:    [Q] node-query indices used
    """
    device = dot_all.device
    H, T, _ = dot_all.shape
    n = T - 1

    dot = dot_all[:, 1:, 1:]
    bias = bias_all[:, 1:, 1:]
    p_base, _ = node_only_conditional(A_all)

    if query_indices is None:
        q_idx = torch.arange(n, device=device)
    else:
        q_idx = torch.tensor(sorted(set(int(q) for q in query_indices if 0 <= int(q) < n)), device=device)
        if q_idx.numel() == 0:
            raise ValueError("No valid query indices after filtering.")

    dot_q = dot[:, q_idx, :]
    bias_q = bias[:, q_idx, :]
    p_q = p_base[:, q_idx, :]
    Q = q_idx.numel()

    uv_list = sample_transpositions(n=n, K=num_perms, seed=seed)

    delta = torch.empty((num_perms, H, Q), device=device, dtype=p_q.dtype)
    inv_tau = 1.0 / max(float(tau), 1e-12)
    for t, (u, v) in enumerate(uv_list):
        delta[t] = (p_q[:, :, u] - p_q[:, :, v]).abs() * inv_tau
    alpha = torch.softmax(delta, dim=0)

    s_struct = torch.zeros((H, Q), device=device, dtype=p_q.dtype)
    s_sym = torch.zeros((H, Q), device=device, dtype=p_q.dtype)

    for t, (u, v) in enumerate(uv_list):
        perm = _swap_perm(n, u, v, device=device)
        logits = dot_q[:, :, perm] + bias_q
        p_pi = torch.softmax(logits.float(), dim=-1).to(p_q.dtype)
        p_perm_ref = p_q[:, :, perm]
        w = alpha[t]
        s_struct += w * cossim_lastdim(p_pi, p_q)
        s_sym += w * cossim_lastdim(p_pi, p_perm_ref)

    return s_struct.detach().cpu().numpy(), s_sym.detach().cpu().numpy(), q_idx.detach().cpu().numpy()


@torch.no_grad()
def kl_scores_one_layer_per_query(
    dot_all: torch.Tensor,
    bias_all: torch.Tensor,
    A_all: torch.Tensor,
    query_indices: list[int] | None = None,
):
    """Per-query normalised KL scores for one layer.

    Returns:
      kl_dot:  [H,Q] = KL(full || dot-only), normalized
      kl_bias: [H,Q] = KL(full || bias-only), normalized
      q_idx:   [Q]
    """
    H, T, _ = A_all.shape
    n = T - 1

    p_full, _ = node_only_conditional(A_all)
    p_dot = torch.softmax(dot_all[:, 1:, 1:].float(), dim=-1)
    p_bias = torch.softmax(bias_all[:, 1:, 1:].float(), dim=-1)

    if query_indices is None:
        q_idx = torch.arange(n, device=A_all.device)
    else:
        q_idx = torch.tensor(sorted(set(int(q) for q in query_indices if 0 <= int(q) < n)), device=A_all.device)
        if q_idx.numel() == 0:
            raise ValueError("No valid query indices after filtering.")

    kl_dot = _normalized_kl(p_full[:, q_idx, :], p_dot[:, q_idx, :]).detach().cpu().numpy()
    kl_bias = _normalized_kl(p_full[:, q_idx, :], p_bias[:, q_idx, :]).detach().cpu().numpy()
    return kl_dot, kl_bias, q_idx.detach().cpu().numpy()


In [ ]:
@torch.no_grad()
def compute_per_query_metrics_for_graph(
    model,
    batch,
    num_perms: int = 64,
    tau: float = 0.02,
    seed: int = 0,
    query_indices: list[int] | None = None,
):
    """Compute per-query scores + KL for all layers in one graph.

    KL outputs are PER-QUERY row divergences: each query row i compares
    full row-distribution A[i,:] against dot-only/bias-only row-distributions.
    """
    dot_list, bias_list, A_list = extract_attention(model, batch)
    L = len(A_list)
    H, T, _ = A_list[0].shape
    n = T - 1

    if query_indices is None:
        Q = n
        q_idx_ref = np.arange(n, dtype=int)
    else:
        q_idx_ref = np.array(sorted(set(int(q) for q in query_indices if 0 <= int(q) < n)), dtype=int)
        Q = len(q_idx_ref)

    s_struct = np.zeros((L, H, Q), dtype=float)
    s_sym = np.zeros((L, H, Q), dtype=float)
    kl_dot = np.zeros((L, H, Q), dtype=float)
    kl_bias = np.zeros((L, H, Q), dtype=float)

    for l in range(L):
        ss, sy, q1 = score_layer_struct_sym_per_query(
            dot_list[l], bias_list[l], A_list[l],
            num_perms=num_perms, tau=tau, seed=seed + l,
            query_indices=(q_idx_ref.tolist() if query_indices is not None else None),
        )
        kd, kb, q2 = kl_scores_one_layer_per_query(
            dot_list[l], bias_list[l], A_list[l],
            query_indices=(q_idx_ref.tolist() if query_indices is not None else None),
        )
        if not np.array_equal(q1, q2):
            raise RuntimeError("Query index mismatch between score and KL computations.")
        s_struct[l] = ss
        s_sym[l] = sy
        kl_dot[l] = kd
        kl_bias[l] = kb

    return {
        "s_struct": s_struct,
        "s_sym": s_sym,
        "kl_dot": kl_dot,
        "kl_bias": kl_bias,
        "query_idx": q1 if L > 0 else q_idx_ref,
        "dot_list": dot_list,
        "bias_list": bias_list,
        "A_list": A_list,
    }


def oxygen_atom_indices(smiles: str) -> list[int]:
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return []
    return [a.GetIdx() for a in mol.GetAtoms() if a.GetSymbol() == "O"]


def central_query_index(dist: np.ndarray, n_nodes: int) -> int:
    """Most-central node by minimum mean shortest-path distance; ties -> smallest idx."""
    D = np.asarray(dist)[:n_nodes, :n_nodes].astype(float)
    D[~np.isfinite(D)] = np.nan
    mean_d = np.nanmean(D, axis=1)
    mean_d = np.where(np.isnan(mean_d), np.inf, mean_d)
    return int(np.argmin(mean_d))


def query_descriptors(smiles: str, dist: np.ndarray):
    """Return per-query descriptors to align analyses across graphs."""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        raise ValueError(f"Bad SMILES: {smiles}")
    n = mol.GetNumAtoms()
    dist_arr = np.asarray(dist)[:n, :n]

    symbols = np.array([a.GetSymbol() for a in mol.GetAtoms()])
    degree = (dist_arr == 1).sum(axis=1).astype(int)

    ecc = np.zeros(n, dtype=float)
    for i in range(n):
        row = dist_arr[i]
        row = row[np.isfinite(row)]
        ecc[i] = float(np.max(row)) if len(row) > 0 else 0.0

    atom_group = np.where(symbols == "O", "O", np.where(symbols == "N", "N", np.where(symbols == "C", "C", "other")))

    q1_d, q2_d = np.quantile(degree, [0.33, 0.66])
    deg_bin = np.where(degree <= q1_d, "low_deg", np.where(degree >= q2_d, "high_deg", "mid_deg"))

    q1_e, q2_e = np.quantile(ecc, [0.33, 0.66])
    ecc_bin = np.where(ecc <= q1_e, "low_ecc", np.where(ecc >= q2_e, "high_ecc", "mid_ecc"))

    return {
        "symbol": symbols,
        "atom_group": atom_group,
        "is_oxygen": (symbols == "O"),
        "degree": degree,
        "ecc": ecc,
        "deg_bin": deg_bin,
        "ecc_bin": ecc_bin,
    }


def safe_corr(x, y, method="pearson"):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    mask = np.isfinite(x) & np.isfinite(y)
    x = x[mask]
    y = y[mask]
    if x.size < 3:
        return np.nan, np.nan
    if np.std(x) < 1e-12 or np.std(y) < 1e-12:
        return np.nan, np.nan
    if method == "spearman":
        r, p = spearmanr(x, y)
    else:
        r, p = pearsonr(x, y)
    return float(r), float(p)


def flatten_with_layer_ids(mat_lh):
    L, H = mat_lh.shape
    vals = mat_lh.reshape(-1)
    layers = np.repeat(np.arange(L), H)
    heads = np.tile(np.arange(H), L)
    return vals, layers, heads


In [ ]:
# (3.1) In-depth single-graph query-variance diagnostics
GRAPH_IDX = 0
NUM_PERMS = 64
TAU = 0.02
smi = smiles_pool[GRAPH_IDX]
batch, n_nodes, dist = build_batch_from_smiles(smi, model=model, device=device)
res_g = compute_per_query_metrics_for_graph(model, batch, num_perms=NUM_PERMS, tau=TAU, seed=0)
desc_g = query_descriptors(smi, dist)
sS = res_g["s_struct"]  # [L,H,Q]
sY = res_g["s_sym"]
q_idx = res_g["query_idx"]
L, H, Q = sS.shape
meanS = sS.mean(axis=-1)
stdS = sS.std(axis=-1)
iqrS = np.percentile(sS, 75, axis=-1) - np.percentile(sS, 25, axis=-1)
meanY = sY.mean(axis=-1)
stdY = sY.std(axis=-1)
iqrY = np.percentile(sY, 75, axis=-1) - np.percentile(sY, 25, axis=-1)
fig, axes = plt.subplots(2, 3, figsize=(17, 9))
for ax, mat, ttl, cmap in [
    (axes[0, 0], meanS, "Structural mean over queries", "YlGnBu"),
    (axes[0, 1], stdS, "Structural std over queries", "YlOrRd"),
    (axes[0, 2], iqrS, "Structural IQR over queries", "magma"),
    (axes[1, 0], meanY, "Symbolic mean over queries", "YlGnBu"),
    (axes[1, 1], stdY, "Symbolic std over queries", "YlOrRd"),
    (axes[1, 2], iqrY, "Symbolic IQR over queries", "magma"),
]:
    im = ax.imshow(mat, aspect="auto", interpolation="nearest", cmap=cmap)
    ax.set_title(ttl, fontsize=11)
    ax.set_xlabel("Head index $h$")
    ax.set_ylabel(r"Layer index $\ell$")
    ax.set_xticks(range(0, H, 4))
    ax.set_yticks(range(L))
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.03)
fig.suptitle(f"Single-graph query variability diagnostics\nSMILES[{GRAPH_IDX}] = {smi}", y=1.02, fontsize=13)
fig.tight_layout()
plt.show()
# Compare node-query average vs CLS-query score for one layer
# Correct pairing: structural <-> bias-only KL surrogate, symbolic <-> dot-only KL surrogate
LAYER_CLS = 1
_, _, A_list = extract_attention(model, batch)
dot_list = res_g["dot_list"]
bias_list = res_g["bias_list"]
cls_struct, cls_sym = score_cls_struct_sym(dot_list[LAYER_CLS], bias_list[LAYER_CLS], A_list[LAYER_CLS], num_perms=96, tau=0.02)
cls_struct = cls_struct.cpu().numpy()
cls_sym = cls_sym.cpu().numpy()
kd = res_g["kl_dot"][LAYER_CLS]   # [H,Q]
kb = res_g["kl_bias"][LAYER_CLS]  # [H,Q]
fig, axes = plt.subplots(1, 3, figsize=(18, 5.2))
ax = axes[0]
ax.scatter(meanS[LAYER_CLS], cls_struct, c=np.arange(H), cmap="viridis", s=30, alpha=0.92)
mn = min(meanS[LAYER_CLS].min(), cls_struct.min()); mx = max(meanS[LAYER_CLS].max(), cls_struct.max())
ax.plot([mn, mx], [mn, mx], "k--", lw=1)
r, p = safe_corr(meanS[LAYER_CLS], cls_struct)
ax.set_title(f"L{LAYER_CLS}: node-query mean vs CLS structural\nr={r:.3f}, p={p:.1e}")
ax.set_xlabel("Node-query mean structural score")
ax.set_ylabel("CLS-query structural score")
ax.grid(True, alpha=0.2)
ax = axes[1]
x = (1.0 - kb[24]) if H > 24 else (1.0 - kb[0])
y = (sS[LAYER_CLS, 24] if H > 24 else sS[LAYER_CLS, 0])
r, p = safe_corr(x, y)
ax.scatter(x, y, s=20, alpha=0.75, color="#2a9d8f")
ax.set_xlim(0.0, 1.0)
ax.set_title(f"L{LAYER_CLS} H{24 if H > 24 else 0}: struct vs (1-KL_bias)\nr={r:.3f}, p={p:.1e}")
ax.set_xlabel(r"$1 - \widetilde{D}_{KL}(A^{full}||A^{bias})$ (per-query row)")
ax.set_ylabel("Structural score (per-query)")
ax.grid(True, alpha=0.2)
ax = axes[2]
x = (1.0 - kd[24]) if H > 24 else (1.0 - kd[0])
y = (sY[LAYER_CLS, 24] if H > 24 else sY[LAYER_CLS, 0])
r, p = safe_corr(x, y)
ax.scatter(x, y, s=20, alpha=0.75, color="#7b2cbf")
ax.set_xlim(0.0, 1.0)
ax.set_title(f"L{LAYER_CLS} H{24 if H > 24 else 0}: sym vs (1-KL_dot)\nr={r:.3f}, p={p:.1e}")
ax.set_xlabel(r"$1 - \widetilde{D}_{KL}(A^{full}||A^{dot})$ (per-query row)")
ax.set_ylabel("Symbolic score (per-query)")
ax.grid(True, alpha=0.2)
fig.tight_layout()
plt.show()
# Detailed query traces for the most query-variable heads
flat_var = stdS.reshape(-1)
top_idx = np.argsort(-flat_var)[:4]
sel = [(i // H, i % H) for i in top_idx]
fig, axes = plt.subplots(2, 2, figsize=(14, 8), sharex=True)
for ax, (l, h) in zip(axes.flatten(), sel):
    xs = np.arange(Q)
    ax.plot(xs, sS[l, h], "o-", ms=3.5, lw=1.3, color="#2a9d8f", label="struct")
    ax.plot(xs, sY[l, h], "s-", ms=3.2, lw=1.2, color="#7b2cbf", label="sym")
    ax.axhline(meanS[l, h], color="#2a9d8f", ls="--", lw=1)
    ax.axhline(meanY[l, h], color="#7b2cbf", ls="--", lw=1)
    oxy = desc_g["is_oxygen"][q_idx]
    if np.any(oxy):
        ax.scatter(xs[oxy], sS[l, h][oxy], c="red", s=28, marker="o", label="oxygen query")
    ax.set_title(f"L{l} H{h} | std={stdS[l,h]:.3f}, iqr={iqrS[l,h]:.3f}")
    ax.set_xlabel("Query index (node order)")
    ax.set_ylabel("Score")
    ax.grid(True, alpha=0.2)
handles, labels = axes.flatten()[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=4, framealpha=0.9, bbox_to_anchor=(0.5, -0.02))
fig.suptitle("Most query-variable heads (single graph)", y=1.02)
fig.tight_layout(rect=(0, 0.05, 1, 1))
plt.show()


In [ ]:
# (3.2) Multi-graph tests: is query-level signal discarded by per-head aggregation?
N_MULTI = 30
NUM_PERMS_MULTI = 40
TAU_MULTI = 0.02
ANCHOR_HEADS = [(1, 24), (11, 14), (0, 0)]

meanS_list, meanY_list = [], []
varS_list, varY_list = [], []
oxy_gap_S, oxy_gap_Y = [], []

# Feature-conditioned residual collections for anchor heads
residual_by_atom = {(l, h): {"O": [], "N": [], "C": [], "other": []} for (l, h) in ANCHOR_HEADS}
residual_by_ecc = {(l, h): {"low_ecc": [], "mid_ecc": [], "high_ecc": []} for (l, h) in ANCHOR_HEADS}

used_graphs = 0
for gi, smi in enumerate(smiles_pool[:N_MULTI]):
    try:
        batch, n_nodes, dist = build_batch_from_smiles(smi, model=model, device=device)
        res = compute_per_query_metrics_for_graph(
            model,
            batch,
            num_perms=NUM_PERMS_MULTI,
            tau=TAU_MULTI,
            seed=10_000 + gi,
        )
        desc = query_descriptors(smi, dist)

        sS = res["s_struct"]
        sY = res["s_sym"]
        q_idx = res["query_idx"]

        mS = sS.mean(axis=-1)
        mY = sY.mean(axis=-1)
        vS = sS.var(axis=-1)
        vY = sY.var(axis=-1)

        meanS_list.append(mS)
        meanY_list.append(mY)
        varS_list.append(vS)
        varY_list.append(vY)

        # Oxygen-vs-nonoxygen per-head score gap
        oxy_mask = desc["is_oxygen"][q_idx]
        non_mask = ~oxy_mask
        if np.any(oxy_mask) and np.any(non_mask):
            oxy_gap_S.append(sS[:, :, oxy_mask].mean(axis=-1) - sS[:, :, non_mask].mean(axis=-1))
            oxy_gap_Y.append(sY[:, :, oxy_mask].mean(axis=-1) - sY[:, :, non_mask].mean(axis=-1))

        # Feature-conditioned residuals at selected heads
        atom_group = desc["atom_group"][q_idx]
        ecc_bin = desc["ecc_bin"][q_idx]
        for (l, h) in ANCHOR_HEADS:
            if l >= sS.shape[0] or h >= sS.shape[1]:
                continue
            resid = sS[l, h] - mS[l, h]
            for grp in ["O", "N", "C", "other"]:
                mask = (atom_group == grp)
                if np.any(mask):
                    residual_by_atom[(l, h)][grp].extend(resid[mask].tolist())
            for grp in ["low_ecc", "mid_ecc", "high_ecc"]:
                mask = (ecc_bin == grp)
                if np.any(mask):
                    residual_by_ecc[(l, h)][grp].extend(resid[mask].tolist())

        used_graphs += 1
    except Exception:
        continue

print(f"Used {used_graphs} graphs for section 3.2")

meanS_arr = np.stack(meanS_list, axis=0)  # [G,L,H]
meanY_arr = np.stack(meanY_list, axis=0)
varS_arr = np.stack(varS_list, axis=0)
varY_arr = np.stack(varY_list, axis=0)

# Variance decomposition: within-query vs between-graph mean
betweenS = meanS_arr.var(axis=0)
betweenY = meanY_arr.var(axis=0)
withinS = varS_arr.mean(axis=0)
withinY = varY_arr.mean(axis=0)
share_within_S = withinS / np.clip(withinS + betweenS, 1e-12, None)
share_within_Y = withinY / np.clip(withinY + betweenY, 1e-12, None)

fig, axes = plt.subplots(1, 2, figsize=(13.5, 5.2))
for ax, mat, ttl in [
    (axes[0], share_within_S, "Within-query variance share (structural)"),
    (axes[1], share_within_Y, "Within-query variance share (symbolic)"),
]:
    im = ax.imshow(mat, aspect="auto", interpolation="nearest", cmap="YlOrRd", vmin=0.0, vmax=1.0)
    ax.set_title(ttl)
    ax.set_xlabel("Head index h")
    ax.set_ylabel("Layer index l")
    ax.set_xticks(range(0, mat.shape[1], 4))
    ax.set_yticks(range(mat.shape[0]))
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.03)

fig.suptitle("How much information is discarded by pooling over queries?", y=1.02)
fig.tight_layout()
plt.show()

# Oxygen-vs-nonoxygen query gaps
if len(oxy_gap_S) > 0:
    mean_oxy_gap_S = np.stack(oxy_gap_S, axis=0).mean(axis=0)
    mean_oxy_gap_Y = np.stack(oxy_gap_Y, axis=0).mean(axis=0)

    fig, axes = plt.subplots(1, 2, figsize=(13.5, 5.2))
    for ax, mat, ttl in [
        (axes[0], mean_oxy_gap_S, "Oxygen query gap (structural): O - nonO"),
        (axes[1], mean_oxy_gap_Y, "Oxygen query gap (symbolic): O - nonO"),
    ]:
        vmax = float(np.nanmax(np.abs(mat)))
        im = ax.imshow(mat, aspect="auto", interpolation="nearest", cmap="coolwarm", vmin=-vmax, vmax=vmax)
        ax.set_title(ttl)
        ax.set_xlabel("Head index h")
        ax.set_ylabel("Layer index l")
        ax.set_xticks(range(0, mat.shape[1], 4))
        ax.set_yticks(range(mat.shape[0]))
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.03)

    fig.tight_layout()
    plt.show()

# Residual distributions by query archetype for anchor heads (structural residuals)
n_cols = len(ANCHOR_HEADS)
fig, axes = plt.subplots(2, n_cols, figsize=(4.8 * n_cols, 8.0), squeeze=False)

for j, (l, h) in enumerate(ANCHOR_HEADS):
    data = residual_by_atom[(l, h)]
    labels = ["O", "N", "C", "other"]
    vals = [data[k] if len(data[k]) > 0 else [np.nan] for k in labels]
    ax = axes[0, j]
    bp = ax.boxplot(vals, labels=labels, showfliers=False, patch_artist=True)
    for patch, c in zip(bp["boxes"], ["#e63946", "#457b9d", "#2a9d8f", "#6c757d"]):
        patch.set_facecolor(c)
        patch.set_alpha(0.6)
    ax.axhline(0, color="k", lw=0.8, ls="--")
    ax.set_title(f"L{l}H{h}: residual by atom group")
    ax.set_ylabel("struct residual (query - head mean)")
    ax.grid(True, axis="y", alpha=0.2)

    data = residual_by_ecc[(l, h)]
    labels = ["low_ecc", "mid_ecc", "high_ecc"]
    vals = [data[k] if len(data[k]) > 0 else [np.nan] for k in labels]
    ax = axes[1, j]
    bp = ax.boxplot(vals, labels=labels, showfliers=False, patch_artist=True)
    for patch, c in zip(bp["boxes"], ["#8ecae6", "#219ebc", "#023047"]):
        patch.set_facecolor(c)
        patch.set_alpha(0.6)
    ax.axhline(0, color="k", lw=0.8, ls="--")
    ax.set_title(f"L{l}H{h}: residual by eccentricity bin")
    ax.set_ylabel("struct residual (query - head mean)")
    ax.grid(True, axis="y", alpha=0.2)

fig.suptitle("Feature-conditioned query residuals (structural score)", y=1.02)
fig.tight_layout()
plt.show()


In [ ]:
# (3.3) KL-vs-score deep dive (raw metrics, differences, and per-query behavior)
N_KL = 25
NUM_PERMS_KL = 32
TAU_KL = 0.02
# Containers for aggregate (per-head, pooled over queries within graph)
aggS_list, aggY_list, aggKD_list, aggKB_list = [], [], [], []
# Containers for per-query global tests
raw_struct_x, raw_struct_y = [], []
raw_sym_x, raw_sym_y = [], []
diff_x, diff_y = [], []
# Per-head/per-layer query-level correlations
vals_struct = None  # nested lists [L][H]
vals_sym = None
vals_diff = None
used = 0
for gi, smi in enumerate(smiles_pool[:N_KL]):
    try:
        batch, n_nodes, dist = build_batch_from_smiles(smi, model=model, device=device)
        res = compute_per_query_metrics_for_graph(
            model,
            batch,
            num_perms=NUM_PERMS_KL,
            tau=TAU_KL,
            seed=40_000 + gi,
        )
        sS = res["s_struct"]
        sY = res["s_sym"]
        kD = res["kl_dot"]
        kB = res["kl_bias"]
        L, H, Q = sS.shape
        # Per-graph aggregate values (matching graph_interp.metrics.scores / .kl style)
        aggS = sS.mean(axis=-1)
        aggY = sY.mean(axis=-1)
        aggKD = kD.mean(axis=-1)
        aggKB = kB.mean(axis=-1)
        aggS_list.append(aggS)
        aggY_list.append(aggY)
        aggKD_list.append(aggKD)
        aggKB_list.append(aggKB)
        # Correct pairing: structural <-> bias-only, symbolic <-> dot-only
        raw_struct_x.append((1.0 - kB).reshape(-1))
        raw_struct_y.append(sS.reshape(-1))
        raw_sym_x.append((1.0 - kD).reshape(-1))
        raw_sym_y.append(sY.reshape(-1))
        diff_x.append((kD - kB).reshape(-1))
        diff_y.append((sS - sY).reshape(-1))
        # Build nested per-LH lists once
        if vals_struct is None:
            vals_struct = [[{"x": [], "y": []} for _ in range(H)] for __ in range(L)]
            vals_sym = [[{"x": [], "y": []} for _ in range(H)] for __ in range(L)]
            vals_diff = [[{"x": [], "y": []} for _ in range(H)] for __ in range(L)]
        for l in range(L):
            for h in range(H):
                vals_struct[l][h]["x"].append((1.0 - kB[l, h]).reshape(-1))
                vals_struct[l][h]["y"].append(sS[l, h].reshape(-1))
                vals_sym[l][h]["x"].append((1.0 - kD[l, h]).reshape(-1))
                vals_sym[l][h]["y"].append(sY[l, h].reshape(-1))
                vals_diff[l][h]["x"].append((kD[l, h] - kB[l, h]).reshape(-1))
                vals_diff[l][h]["y"].append((sS[l, h] - sY[l, h]).reshape(-1))
        used += 1
    except Exception:
        continue
print(f"Used {used} graphs for KL deep-dive")
mean_struct = np.stack(aggS_list, axis=0).mean(axis=0)
mean_sym = np.stack(aggY_list, axis=0).mean(axis=0)
mean_kl_dot = np.stack(aggKD_list, axis=0).mean(axis=0)
mean_kl_bias = np.stack(aggKB_list, axis=0).mean(axis=0)
# A) Aggregate-level relationships (mirrors and extends Section I)
fig, axes = plt.subplots(2, 2, figsize=(14.5, 11.2))
x, layers, heads = flatten_with_layer_ids(1.0 - mean_kl_bias)
y, _, _ = flatten_with_layer_ids(mean_struct)
r, p = safe_corr(x, y)
sc = axes[0, 0].scatter(x, y, c=layers, cmap="viridis", s=24, alpha=0.85)
axes[0, 0].set_title(f"Aggregate structural vs 1-KL_bias\nr={r:.3f}, p={p:.1e}", fontsize=11)
axes[0, 0].set_xlabel(r"$1 - \widetilde{D}_{KL}(A^{full}||A^{bias})$")
axes[0, 0].set_ylabel(r"$\bar{s}_{struct}$")
axes[0, 0].set_xlim(0.0, 1.0)
axes[0, 0].set_ylim(0.0, 1.0)
axes[0, 0].grid(True, alpha=0.2)
fig.colorbar(sc, ax=axes[0, 0], fraction=0.046, pad=0.03, label=r"Layer $\ell$")
x, layers, heads = flatten_with_layer_ids(1.0 - mean_kl_dot)
y, _, _ = flatten_with_layer_ids(mean_sym)
r, p = safe_corr(x, y)
sc = axes[0, 1].scatter(x, y, c=layers, cmap="viridis", s=24, alpha=0.85)
axes[0, 1].set_title(f"Aggregate symbolic vs 1-KL_dot\nr={r:.3f}, p={p:.1e}", fontsize=11)
axes[0, 1].set_xlabel(r"$1 - \widetilde{D}_{KL}(A^{full}||A^{dot})$")
axes[0, 1].set_ylabel(r"$\bar{s}_{sym}$")
axes[0, 1].set_xlim(0.0, 1.0)
axes[0, 1].set_ylim(0.0, 1.0)
axes[0, 1].grid(True, alpha=0.2)
fig.colorbar(sc, ax=axes[0, 1], fraction=0.046, pad=0.03, label=r"Layer $\ell$")
diff_score = mean_struct - mean_sym
diff_kl = mean_kl_dot - mean_kl_bias
x, layers, heads = flatten_with_layer_ids(diff_kl)
y, _, _ = flatten_with_layer_ids(diff_score)
r, p = safe_corr(x, y)
sc = axes[1, 0].scatter(x, y, c=layers, cmap="viridis", s=24, alpha=0.85)
axes[1, 0].set_title(f"Aggregate difference relation\nr={r:.3f}, p={p:.1e}", fontsize=11)
axes[1, 0].set_xlabel(r"$\widetilde{D}_{KL}^{dot} - \widetilde{D}_{KL}^{bias}$")
axes[1, 0].set_ylabel(r"$\bar{s}_{struct} - \bar{s}_{sym}$")
axes[1, 0].grid(True, alpha=0.2)
fig.colorbar(sc, ax=axes[1, 0], fraction=0.046, pad=0.03, label=r"Layer $\ell$")
# Per-layer correlation profile
L = mean_struct.shape[0]
layers_arr = np.arange(L)
r_struct_layer = np.array([safe_corr(1.0 - mean_kl_bias[l], mean_struct[l])[0] for l in range(L)])
r_sym_layer = np.array([safe_corr(1.0 - mean_kl_dot[l], mean_sym[l])[0] for l in range(L)])
r_diff_layer = np.array([safe_corr(diff_kl[l], diff_score[l])[0] for l in range(L)])
axes[1, 1].plot(layers_arr, r_struct_layer, "o-", label="struct vs (1-KL_bias)", color="#264653", lw=2)
axes[1, 1].plot(layers_arr, r_sym_layer, "s-", label="sym vs (1-KL_dot)", color="#e76f51", lw=2)
axes[1, 1].plot(layers_arr, r_diff_layer, "^-", label="difference vs KL difference", color="#6a4c93", lw=2)
axes[1, 1].axhline(0, color="k", lw=0.9, ls="--")
axes[1, 1].set_xticks(layers_arr)
axes[1, 1].set_ylim(-1.05, 1.05)
axes[1, 1].set_xlabel(r"Layer index $\ell$")
axes[1, 1].set_ylabel("Pearson $r$ across heads")
axes[1, 1].set_title("Per-layer correlation profile", fontsize=11)
axes[1, 1].legend(fontsize=9, framealpha=0.9)
axes[1, 1].grid(True, alpha=0.2)
fig.suptitle("KL vs score relationships: aggregate level", y=1.02, fontsize=13)
fig.tight_layout()
plt.show()
# B) Per-query global relationships
xS = np.concatenate(raw_struct_x)
yS = np.concatenate(raw_struct_y)
xY = np.concatenate(raw_sym_x)
yY = np.concatenate(raw_sym_y)
xD = np.concatenate(diff_x)
yD = np.concatenate(diff_y)
fig, axes = plt.subplots(1, 3, figsize=(18.5, 5.5))
for ax, x, y, title, color, xlab, ylab in [
    (axes[0], xS, yS, "Per-query structural vs 1-KL_bias", "#264653", r"$1 - \widetilde{D}_{KL}(A^{full}||A^{bias})$", "Structural score"),
    (axes[1], xY, yY, "Per-query symbolic vs 1-KL_dot", "#b56576", r"$1 - \widetilde{D}_{KL}(A^{full}||A^{dot})$", "Symbolic score"),
    (axes[2], xD, yD, "Per-query difference vs KL difference", "#6a4c93", r"$\widetilde{D}_{KL}^{dot} - \widetilde{D}_{KL}^{bias}$", r"$s_{struct}-s_{sym}$"),
]:
    n = len(x)
    keep = np.arange(n)
    if n > 70000:
        rng = np.random.default_rng(0)
        keep = rng.choice(n, size=70000, replace=False)
    xr, yr = x[keep], y[keep]
    r, p = safe_corr(xr, yr)
    ax.scatter(xr, yr, s=6, alpha=0.18, color=color)
    ax.set_title(f"{title}\nr={r:.3f}, p={p:.1e}", fontsize=11)
    ax.set_xlabel(xlab)
    ax.set_ylabel(ylab)
    if "KL}^{dot} -" not in xlab:
        ax.set_xlim(0.0, 1.0)
        ax.set_ylim(0.0, 1.0)
    ax.grid(True, alpha=0.2)
fig.tight_layout()
plt.show()
# C) Per-LH query-level correlation heatmaps
corr_struct_lh = np.full((L, H), np.nan)
corr_sym_lh = np.full((L, H), np.nan)
corr_diff_lh = np.full((L, H), np.nan)
for l in range(L):
    for h in range(H):
        xs = np.concatenate(vals_struct[l][h]["x"]) if len(vals_struct[l][h]["x"]) else np.array([])
        ys = np.concatenate(vals_struct[l][h]["y"]) if len(vals_struct[l][h]["y"]) else np.array([])
        corr_struct_lh[l, h] = safe_corr(xs, ys)[0]
        xs = np.concatenate(vals_sym[l][h]["x"]) if len(vals_sym[l][h]["x"]) else np.array([])
        ys = np.concatenate(vals_sym[l][h]["y"]) if len(vals_sym[l][h]["y"]) else np.array([])
        corr_sym_lh[l, h] = safe_corr(xs, ys)[0]
        xs = np.concatenate(vals_diff[l][h]["x"]) if len(vals_diff[l][h]["x"]) else np.array([])
        ys = np.concatenate(vals_diff[l][h]["y"]) if len(vals_diff[l][h]["y"]) else np.array([])
        corr_diff_lh[l, h] = safe_corr(xs, ys)[0]
fig, axes = plt.subplots(1, 3, figsize=(18.5, 5.5))
for ax, mat, title in [
    (axes[0], corr_struct_lh, "Query-level corr: struct vs 1-KL_bias"),
    (axes[1], corr_sym_lh, "Query-level corr: sym vs 1-KL_dot"),
    (axes[2], corr_diff_lh, "Query-level corr: difference vs KL difference"),
]:
    im = ax.imshow(mat, aspect="auto", interpolation="nearest", cmap="coolwarm", vmin=-1, vmax=1)
    ax.set_title(title, fontsize=11)
    ax.set_xlabel("Head index $h$")
    ax.set_ylabel(r"Layer index $\ell$")
    ax.set_xticks(range(0, H, 4))
    ax.set_yticks(range(L))
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.03)
fig.suptitle("Where do KL proxies align (or fail) at query level?", y=1.02, fontsize=13)
fig.tight_layout()
plt.show()
# D) Recreate Section-I figures here for direct side-by-side reference
fig = fig39_structural_dom_vs_kl_diff(mean_struct, mean_sym, mean_kl_dot, mean_kl_bias, N=used)
plt.show()
fig = fig40_kl_disagreement(mean_struct, mean_sym, mean_kl_dot, mean_kl_bias, N=used)
plt.show()
fig = fig41_signed_bias(mean_struct, mean_sym, mean_kl_dot, mean_kl_bias, N=used)
plt.show()


In [ ]:
# (3.4) Most-central-node query vs all-query aggregate across graphs
N_CENTRAL = 40
NUM_PERMS_CENTRAL = 40
TAU_CENTRAL = 0.02
allS_list, allY_list = [], []
ctrS_list, ctrY_list = [], []
used_ctr = 0
for gi, smi in enumerate(smiles_pool[:N_CENTRAL]):
    try:
        batch, n_nodes, dist = build_batch_from_smiles(smi, model=model, device=device)
        res_all = compute_per_query_metrics_for_graph(
            model, batch,
            num_perms=NUM_PERMS_CENTRAL,
            tau=TAU_CENTRAL,
            seed=80_000 + gi,
            query_indices=None,
        )
        cidx = central_query_index(dist, n_nodes)
        res_ctr = compute_per_query_metrics_for_graph(
            model, batch,
            num_perms=NUM_PERMS_CENTRAL,
            tau=TAU_CENTRAL,
            seed=90_000 + gi,
            query_indices=[cidx],
        )
        allS_list.append(res_all["s_struct"].mean(axis=-1))
        allY_list.append(res_all["s_sym"].mean(axis=-1))
        ctrS_list.append(res_ctr["s_struct"].squeeze(-1))
        ctrY_list.append(res_ctr["s_sym"].squeeze(-1))
        used_ctr += 1
    except Exception:
        continue
print(f"Used {used_ctr} graphs for central-node test")
mean_allS = np.stack(allS_list, axis=0).mean(axis=0)
mean_allY = np.stack(allY_list, axis=0).mean(axis=0)
mean_ctrS = np.stack(ctrS_list, axis=0).mean(axis=0)
mean_ctrY = np.stack(ctrY_list, axis=0).mean(axis=0)
deltaS = mean_ctrS - mean_allS
deltaY = mean_ctrY - mean_allY
fig, axes = plt.subplots(2, 3, figsize=(17, 9))
for ax, mat, title, cmap, vv in [
    (axes[0, 0], mean_allS, "All-query aggregate: structural", "YlGnBu", None),
    (axes[0, 1], mean_ctrS, "Central-node only: structural", "YlGnBu", None),
    (axes[0, 2], deltaS, "Central - all (structural)", "coolwarm", "sym"),
    (axes[1, 0], mean_allY, "All-query aggregate: symbolic", "YlGnBu", None),
    (axes[1, 1], mean_ctrY, "Central-node only: symbolic", "YlGnBu", None),
    (axes[1, 2], deltaY, "Central - all (symbolic)", "coolwarm", "sym"),
]:
    if vv == "sym":
        v = float(np.nanmax(np.abs(mat)))
        im = ax.imshow(mat, aspect="auto", interpolation="nearest", cmap=cmap, vmin=-v, vmax=v)
    else:
        im = ax.imshow(mat, aspect="auto", interpolation="nearest", cmap=cmap)
    ax.set_title(title, fontsize=11)
    ax.set_xlabel("Head index $h$")
    ax.set_ylabel(r"Layer index $\ell$")
    ax.set_xticks(range(0, mat.shape[1], 4))
    ax.set_yticks(range(mat.shape[0]))
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.03)
fig.suptitle("Most-central query vs pooled all-query scores", y=1.02, fontsize=13)
fig.tight_layout()
plt.show()
# Correlation + effect size summaries
xS = mean_allS.reshape(-1); yS = mean_ctrS.reshape(-1)
xY = mean_allY.reshape(-1); yY = mean_ctrY.reshape(-1)
rS, pS = safe_corr(xS, yS)
rY, pY = safe_corr(xY, yY)
madS = np.abs(deltaS).mean(); madY = np.abs(deltaY).mean()
maxS = np.abs(deltaS).max();  maxY = np.abs(deltaY).max()
fig, axes = plt.subplots(1, 2, figsize=(12.5, 5.0))
ax = axes[0]
ax.scatter(xS, yS, s=18, alpha=0.8, color="#2a9d8f")
mn, mx = min(xS.min(), yS.min()), max(xS.max(), yS.max())
ax.plot([mn, mx], [mn, mx], 'k--', lw=1)
ax.set_title(f"Structural: central vs all\nr={rS:.3f}, p={pS:.1e}")
ax.set_xlabel("All-query aggregate")
ax.set_ylabel("Central-node only")
ax.grid(True, alpha=0.2)
ax = axes[1]
ax.scatter(xY, yY, s=18, alpha=0.8, color="#7b2cbf")
mn, mx = min(xY.min(), yY.min()), max(xY.max(), yY.max())
ax.plot([mn, mx], [mn, mx], 'k--', lw=1)
ax.set_title(f"Symbolic: central vs all\nr={rY:.3f}, p={pY:.1e}")
ax.set_xlabel("All-query aggregate")
ax.set_ylabel("Central-node only")
ax.grid(True, alpha=0.2)
fig.suptitle(
    f"Deviation summary | mean |Δ|: struct={madS:.4f}, sym={madY:.4f} | max |Δ|: struct={maxS:.4f}, sym={maxY:.4f}",
    y=1.03,
    fontsize=11,
)
fig.tight_layout()
plt.show()
